# GA4 E-commerce Conversion Drivers: models and experiment design

Runs in Google Colab against BigQuery project `aha-moment-analysis`.

1. Load the clean feature table
2. Explanatory model: does viewing a product matter after controlling for engagement, device and channel?
3. Power analysis: how long would an A/B test take?
4. Predictive model: train on November users, test on December users

Results from the original run are shown in markdown under each step.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project="aha-moment-analysis")

df = client.query(
    "SELECT * FROM `aha-moment-analysis.analysis.user_features_clean`"
).to_dataframe()

print(df.shape)
print("Conversion rate:", round(df["converted_30d"].mean() * 100, 2), "%")

Result: `(167441, 18)`, conversion rate **1.57%**.

## 2. Explanatory model (logistic regression)
Compares users with the same engagement time, page views, device and traffic source.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df["y"]               = df["converted_30d"].astype(int)
df["viewed_product"]  = (df["s1_items_viewed"] > 0).astype(int)
df["clicked_product"] = (df["s1_items_clicked"] > 0).astype(int)
df["used_search"]     = (df["s1_searches"] > 0).astype(int)
df["clicked_promo"]   = (df["s1_promo_clicks"] > 0).astype(int)
df["log_engaged_sec"] = np.log1p(df["s1_engaged_sec"])
df["log_page_views"]  = np.log1p(df["s1_page_views"])

model = smf.logit(
    "y ~ viewed_product + clicked_product + used_search + clicked_promo"
    " + log_engaged_sec + log_page_views + C(device) + C(medium)",
    data=df
).fit()

results = pd.DataFrame({
    "odds_ratio": np.exp(model.params),
    "p_value": model.pvalues
}).round(3)
print(results)

Result (odds ratios, all behaviour terms p < 0.001):

| Term | Odds ratio |
| --- | --- |
| viewed_product | **2.160** |
| clicked_product | 0.747 |
| used_search | 0.659 |
| clicked_promo | 0.598 |
| log_engaged_sec | 1.534 |
| log_page_views | 3.792 |
| device / medium | not significant (p > 0.2) |

Viewing a product doubles the odds of buying even among otherwise similar users. Search and promo clicks are linked to *lower* odds once engagement is controlled.

## 3. Power analysis
How many users does an A/B test need to detect a 10% relative lift?

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

daily_new_users = 168498 / 61   # new users per day in the cohort window

# (a) Primary metric = 30-day conversion
baseline = 0.0157
effect = proportion_effectsize(baseline * 1.10, baseline)
n_per_group = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8, ratio=1)
print("Conversion  -> per group:", round(n_per_group), "| total:", round(n_per_group * 2),
      "| days:", round((n_per_group * 2) / daily_new_users))

# (b) Primary metric = first-session product-view rate (leading indicator)
baseline_view = 0.202
effect_view = proportion_effectsize(baseline_view * 1.10, baseline_view)
n_view = NormalIndPower().solve_power(effect_size=effect_view, alpha=0.05, power=0.8, ratio=1)
print("View rate   -> per group:", round(n_view), "| total:", round(n_view * 2),
      "| days:", round((n_view * 2) / daily_new_users, 1))

Result:
- 30-day conversion: 103,196 per group, 206,392 total, **~75 days**
- Product-view rate: 6,426 per group, 12,852 total, **~4.7 days**

Recommendation: use product-view rate as the primary metric and run for 2 full weeks to cover weekly cycles; keep 30-day conversion and revenue per user as guardrails.

## 4. Predictive model with a time-based split
Train on November users, test on December users.

In [ ]:
from google.cloud import bigquery
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

client = bigquery.Client(project="aha-moment-analysis")
df = client.query("""
SELECT
  first_week, device, medium,
  s1_page_views, s1_scrolls, s1_items_viewed, s1_items_clicked,
  s1_searches, s1_promo_views, s1_promo_clicks, s1_engaged_sec,
  CAST(converted_30d AS INT64) AS y
FROM `aha-moment-analysis.analysis.user_features_clean`
""").to_dataframe()

# --- features: only first-session behaviour (no cart / checkout, to avoid leakage) ---
num_cols = ["s1_page_views", "s1_scrolls", "s1_items_viewed", "s1_items_clicked",
            "s1_searches", "s1_promo_views", "s1_promo_clicks", "s1_engaged_sec"]
X = np.log1p(df[num_cols].fillna(0).astype(float))
X["viewed_product"] = (df["s1_items_viewed"] > 0).astype(int)
X = pd.concat([X, pd.get_dummies(df[["device", "medium"]].fillna("unknown"), drop_first=True, dtype=int)], axis=1)
y = df["y"]

# --- time-based split: learn from November, predict December ---
month = pd.to_datetime(df["first_week"]).dt.month
train, test = month == 11, month == 12
print(f"Train (Nov): {train.sum():,} users, {y[train].mean():.2%} buy")
print(f"Test  (Dec): {test.sum():,} users, {y[test].mean():.2%} buy")

scaler = StandardScaler().fit(X[train])
models = {
    "Logistic regression": LogisticRegression(max_iter=2000).fit(scaler.transform(X[train]), y[train]),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=42).fit(X[train], y[train]),
}

def evaluate(name, scores):
    # Treat the top 10% highest-scored December users as "likely buyers"
    cutoff = np.quantile(scores, 0.90)
    pred = (scores >= cutoff).astype(int)
    return {
        "model": name,
        "AUC": round(roc_auc_score(y[test], scores), 3),
        "PR-AUC": round(average_precision_score(y[test], scores), 3),
        "precision@top10%": round(precision_score(y[test], pred), 3),
        "recall@top10%": round(recall_score(y[test], pred), 3),
    }

results = [
    evaluate("Logistic regression", models["Logistic regression"].predict_proba(scaler.transform(X[test]))[:, 1]),
    evaluate("Gradient boosting", models["Gradient boosting"].predict_proba(X[test])[:, 1]),
]

# Simple baseline: "viewed a product" rule (no model)
rule = X.loc[test, "viewed_product"]
results.append({
    "model": "Rule: viewed a product",
    "AUC": round(roc_auc_score(y[test], rule), 3),
    "PR-AUC": round(average_precision_score(y[test], rule), 3),
    "precision@top10%": round(precision_score(y[test], rule), 3),
    "recall@top10%": round(recall_score(y[test], rule), 3),
})

print(pd.DataFrame(results).to_string(index=False))
print(f"\nBase rate in December (random guessing precision): {y[test].mean():.3f}")

Result:

| Model | AUC | PR-AUC | Precision (top 10%) | Recall (top 10%) |
| --- | --- | --- | --- | --- |
| Logistic regression | 0.937 | 0.369 | 0.112 | 0.823 |
| Gradient boosting | 0.943 | 0.349 | 0.112 | 0.824 |
| Rule: viewed a product | 0.843 | 0.058 | 0.066 | 0.851 |

Train: 86,631 users (1.76% buy). Test: 80,810 users (1.36% buy). Logistic regression is preferred: same performance, fully explainable.